In [ ]:
# -*- coding:utf-8 -*-
"""
作业：ElasticNet 分析Netflix评分数据
仓库：与上周Ridge Lasso作业同一仓库
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error

# ====================== 1. 数据说明 ======================
"""
Kaggle Netflix Prize数据集，原始数据巨大，下载后，这里使用抽样子集。
下载地址：kaggle Netflix prize
⚠️ 不要把大csv提交git！
我们做特征工程，不直接使用稀疏评分矩阵。

字段说明示例：
user_id, movie_id, rating, date
"""

# -------- 读取数据，请把路径改成你本地文件路径 --------
df_raw = pd.read_csv("netflix_sample.csv")
print(df_raw.shape)
df_raw.head()

# ====================== 2. 数据清洗 & 特征工程 ======================
# 统计每一个用户的统计特征
user_stat = df_raw.groupby("user_id").agg(
    user_mean_rating=("rating", "mean"),
    user_std_rating=("rating", "std"),
    user_cnt=("rating", "count")
).reset_index()

# 统计每一部电影的统计特征
movie_stat = df_raw.groupby("movie_id").agg(
    movie_mean_rating=("rating", "mean"),
    movie_std_rating=("rating", "std"),
    movie_cnt=("rating", "count")
).reset_index()

# merge 构造回归数据集
df = df_raw.merge(user_stat, on="user_id")
df = df.merge(movie_stat, on="movie_id")

# 目标变量 y：用户真实评分
y = df["rating"].copy()

# 构造特征X：去掉id、date、target
feature_cols = ["user_mean_rating", "user_std_rating", "user_cnt",
                "movie_mean_rating", "movie_std_rating", "movie_cnt"]
X = df[feature_cols].copy()

# 划分训练集、测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# ⭐正则化必须标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ======================3. ElasticNetCV 交叉验证 ======================
# alpha：λ； l1_ratio 控制L1占比：0等价Ridge，1等价Lasso
l1_ratios_grid = [0.1,0.3,0.5,0.7,0.9]
alphas_grid = np.logspace(-4, 2, 100)

en_cv = ElasticNetCV(
    alphas=alphas_grid,
    l1_ratio=l1_ratios_grid,
    cv=10,
    max_iter=20000,
    n_jobs=-1,
    random_state=42
)
en_cv.fit(X_train_scaled, y_train)

print("="*50)
print(f"最佳 l1_ratio: {en_cv.l1_ratio_:.3f}")
print(f"最佳 alpha(λ): {en_cv.alpha_:.5f}")

# 预测
y_pred_train = en_cv.predict(X_train_scaled)
y_pred_test = en_cv.predict(X_test_scaled)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f"Train RMSE: {rmse_train:.4f}")
print(f"Test RMSE:  {rmse_test:.4f}")

# 统计非零系数
coef_series = pd.Series(en_cv.coef_, index=feature_cols)
non_zero_coef = coef_series[coef_series.abs() > 1e-6]
print("\n非零系数：")
print(non_zero_coef)
print(f"\n非零特征数量：{len(non_zero_coef)} / {len(feature_cols)}")

# ======================4. 对比 Ridge、Lasso ======================
# Ridge
ridge_cv = RidgeCV(alphas=alphas_grid, cv=10).fit(X_train_scaled, y_train)
# Lasso
lasso_cv = LassoCV(alphas=alphas_grid, cv=10, max_iter=20000).fit(X_train_scaled, y_train)

y_pred_ridge = ridge_cv.predict(X_test_scaled)
y_pred_lasso = lasso_cv.predict(X_test_scaled)

print("="*50)
print(f"Ridge best alpha: {ridge_cv.alpha_:.4f} Test RMSE: {np.sqrt(mean_squared_error(y_test,y_pred_ridge)):.4f}")
print(f"Lasso best alpha: {lasso_cv.alpha_:.4f} Test RMSE: {np.sqrt(mean_squared_error(y_test,y_pred_lasso)):.4f}")

# ======================5. 绘制系数路径 ======================
from sklearn.linear_model import ElasticNet
coef_path = []
alpha_list = np.logspace(-4,1,50)
best_l1 = en_cv.l1_ratio_

for a in alpha_list:
    m = ElasticNet(alpha=a, l1_ratio=best_l1, max_iter=30000)
    m.fit(X_train_scaled, y_train)
    coef_path.append(m.coef_.copy())

coef_path = np.array(coef_path)

plt.figure(figsize=(10,6))
for idx, col in enumerate(feature_cols):
    plt.plot(np.log10(alpha_list), coef_path[:,idx], label=col)
plt.axvline(np.log10(en_cv.alpha_), linestyle="--", color="red", label="best α")
plt.xlabel("log10(alpha)")
plt.ylabel("系数值")
plt.title(f"ElasticNet系数路径 l1_ratio={best_l1:.2f}")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ======================6. 1‑SE法则说明与分析 ======================
"""
ElasticNetCV不直接返回1‑SE点。
两种方案：
1. 手动读取cv_scores_，实现1‑SE法则，选择MSE在最小MSE+1std范围内最大alpha，获得更稀疏模型。
2. 文字讨论：当前最优模型是否过于复杂？是否可以增大alpha换取更少特征、轻微损失性能。
"""

# ======================7. 业务分析与结论 ======================
"""
分析点写在markdown cell：
1. ElasticNet选择的l1_ratio偏向L1还是偏向L2？说明数据更适合Lasso还是Ridge？
2. 哪些特征对电影评分预测影响最大？（看系数正负）
3. ElasticNet对比Ridge/Lasso测试集RMSE的优劣；
4. 局限性：我们只用了用户、电影统计特征，没有使用ID本身；原始Netflix是高度稀疏矩阵；ElasticNet回归不是专门做矩阵补全的，和SVD矩阵分解推荐系统的区别。
5. 改进方向：隐因子矩阵分解（SVD课件中学到的方法）。
"""
